# AI Meeting Notes Analyzer using LangGraph
Implementation using **LangGraph** and **Google Gemini (`gemini-3.5-flash-lite`)**.

In [6]:
import os
from typing import TypedDict
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph, START, END

load_dotenv()

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite", google_api_key=os.getenv("GOOGLE_API_KEY")
)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [7]:
class MeetingState(TypedDict):
    transcript: str
    topics: str
    summary: str
    action_items: str
    owners: str
    priority: str
    final_report: str

In [8]:
def input_node(state: MeetingState):
    return {"transcript": state["transcript"]}


def topic_agent(state: MeetingState):
    prompt = f"Identify the main discussion topics as a numbered list from this transcript:\n{state['transcript']}"
    return {"topics": llm.invoke(prompt).text.strip()}


def summary_agent(state: MeetingState):
    prompt = f"Write a short meeting summary (3 to 5 sentences) for this transcript:\n{state['transcript']}"
    return {"summary": llm.invoke(prompt).text.strip()}


def action_item_agent(state: MeetingState):
    prompt = (
        "Extract action items from this transcript as a numbered list (Task - Owner). "
        "If the owner is not mentioned, label it as 'Owner: Not specified'. "
        "If there are no action items, reply with exactly: No action items identified in this meeting.\n"
        f"{state['transcript']}"
    )
    actions = llm.invoke(prompt).text.strip()

    if "No action items" in actions:
        return {
            "action_items": "No action items identified in this meeting.",
            "owners": "None",
        }

    owner_prompt = (
        f"List the task owners for each task from these action items:\n{actions}"
    )
    owners = llm.invoke(owner_prompt).text.strip()
    return {"action_items": actions, "owners": owners}


def priority_agent(state: MeetingState):
    prompt = (
        "Determine the priority level of these tasks (High Priority, Medium Priority, or Low Priority) "
        "based on urgency phrases like 'urgent', 'ASAP', 'by tomorrow', 'this week', or 'before Friday'. "
        "Return only the priority level.\n"
        f"Transcript:\n{state['transcript']}\nTasks:\n{state['action_items']}"
    )
    return {"priority": llm.invoke(prompt).text.strip()}


def check_action_items(state: MeetingState):
    if "No action items" in state["action_items"]:
        return "final_output"
    return "priority_agent"


def final_output(state: MeetingState):
    priority = state.get(
        "priority", "Skipped (No action items identified in this meeting.)"
    )
    report = (
        f"Meeting Summary\n{state['summary']}\n\n"
        f"Key Topics\n{state['topics']}\n\n"
        f"Action Items\n{state['action_items']}\n\n"
        f"Task Owners\n{state['owners']}\n\n"
        f"Priority\n{priority}"
    )
    return {"final_report": report}

In [9]:
workflow = StateGraph(MeetingState)

workflow.add_node("input_node", input_node)
workflow.add_node("topic_agent", topic_agent)
workflow.add_node("summary_agent", summary_agent)
workflow.add_node("action_item_agent", action_item_agent)
workflow.add_node("priority_agent", priority_agent)
workflow.add_node("final_output", final_output)

workflow.add_edge(START, "input_node")
workflow.add_edge("input_node", "topic_agent")
workflow.add_edge("topic_agent", "summary_agent")
workflow.add_edge("summary_agent", "action_item_agent")

workflow.add_conditional_edges(
    "action_item_agent",
    check_action_items,
    {"priority_agent": "priority_agent", "final_output": "final_output"},
)

workflow.add_edge("priority_agent", "final_output")
workflow.add_edge("final_output", END)

app = workflow.compile()

In [10]:
transcript_input = """John: We need to improve the website performance.
Sarah: Yes, page load time is too slow.
David: I will optimize the database queries this week.
Sarah: I will redesign the homepage layout.
John: Let's try to finish these tasks before Friday."""

result = app.invoke({"transcript": transcript_input})
print(result["final_report"])

Meeting Summary
The team met to address the website's slow page load times and overall performance issues. To resolve this, David agreed to optimize the database queries, while Sarah will redesign the homepage layout. The team set a goal to complete both tasks before the end of the week.

Key Topics
Here are the main discussion topics from the transcript:

1. Website performance improvement (specifically page load time)
2. Database query optimization (assigned to David)
3. Homepage layout redesign (assigned to Sarah)
4. Project deadline (targeting this Friday)

Action Items
1. Optimize the database queries - Owner: David
2. Redesign the homepage layout - Owner: Sarah

Task Owners
Here are the task owners for each task:

1. **Optimize the database queries** - David
2. **Redesign the homepage layout** - Sarah

Priority
1. High Priority
2. High Priority


In [11]:
transcript_input = """Alex: We are getting close to the mobile app release, but the login screen bug is still open.
Priya: Yes, we also need the release notes written soon.
Alex: I will update the API endpoints by tomorrow.
Priya: Great, and someone will need to write the release notes for the mobile app."""

result = app.invoke({"transcript": transcript_input})
print(result["final_report"])

Meeting Summary
The team discussed remaining tasks ahead of the upcoming mobile app release, noting that the login screen bug is still open and release notes need to be written. Alex committed to updating the API endpoints by tomorrow. Finally, Priya highlighted the need for someone to draft the release notes for the app.

Key Topics
Here are the main discussion topics from the transcript:

1. The status of the mobile app release and the open login screen bug.
2. The need to write the release notes for the mobile app.
3. Updating the API endpoints.

Action Items
1. Update the API endpoints - Owner: Alex
2. Write the release notes for the mobile app - Owner: Not specified

Task Owners
Here are the task owners for each action item:

1. Update the API endpoints - **Alex**
2. Write the release notes for the mobile app - **Not specified**

Priority
1. High Priority
2. Medium Priority


In [12]:
transcript_input = """Manager: Congratulations everyone on a successful Q3 product launch!
Lisa: Customers have shared very positive feedback about the new dashboard.
Mark: Our user signups increased by 25% over the past month.
Manager: Thank you all for your hard work. Enjoy the rest of your Friday!"""

result = app.invoke({"transcript": transcript_input})
print(result["final_report"])

Meeting Summary
The team celebrated a successful Q3 product launch, driven by positive customer feedback on the new dashboard. Mark highlighted a 25% increase in user signups over the past month. The manager concluded the meeting by thanking everyone for their hard work and wishing them a good weekend.

Key Topics
Here are the main discussion topics from the transcript:

1. Q3 product launch success and team appreciation
2. Positive customer feedback regarding the new dashboard
3. Increase in user signups (25% over the past month)

Action Items
No action items identified in this meeting.

Task Owners
None

Priority
Skipped (No action items identified in this meeting.)
